# Bayesian Networks

## Core Idea

A Bayesian network consists of:

- A directed acyclic graph, or DAG.
- One conditional probability distribution for each variable.

Example: Suppose we have: Weather $\rightarrow$ Traffic $\rightarrow$ Delay 

and also: Weather $\rightarrow$ Delay

The graph says:

- traffic depends on weather;
- delay depends on weather and traffic;
- weather has no parents.

The joint distribution factorizes as: $$P(W,T,D) = P(W) P(T|W) P(D|W,T)$$

Instead of storing one large joint distribution, we store three smaller factors: $$\phi_W(W)=P(W)$$
$$\phi_T(W,T)=P(T|W)$$
$$\phi_D(W,T,D)=P(D|W,T)$$

Their product reconstructs the complete joint distribution.

In [11]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Hashable, Mapping, Sequence

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
#%run 05_DiscreteFactor.ipynb

In [14]:
# %%capture
# %run "05_DiscreteFactor.ipynb"

In [10]:
@dataclass
class BayesianNetwork:
    """
    Discrete Bayesian network represented as a directed acyclic graph.

    Each node has:
    - a finite domain;
    - zero or more parent nodes;
    - a conditional probability factor.
    """
    domains: Mapping[str, Sequence[Hashable]]
    name: str = "Bayesian Network"

    graph: nx.DiGraph = field(
        init=False,
        repr=False,
    )

    cpts: dict[str, Factor] = field(
        init=False,
        repr=False,
    )

    def __post_init__(self) -> None:
        self.domains = {
            variable: np.asarray(domain, dtype=object)
            for variable, domain in self.domains.items()
        }

        self._validate_domains()

        self.graph = nx.DiGraph()
        self.graph.add_nodes_from(self.domains)

        self.cpts = {}

    def _validate_domains(self) -> None:
        """Validate the domains of all network variables."""
    
        if len(self.domains) == 0:
            raise ValueError(
                "A Bayesian network must contain at least one variable."
            )
    
        for variable, domain in self.domains.items():
            if not isinstance(variable, str):
                raise TypeError(
                    "Variable names must be strings."
                )
    
            if domain.ndim != 1:
                raise ValueError(
                    f"The domain of {variable!r} must be "
                    "one-dimensional."
                )
    
            if len(domain) == 0:
                raise ValueError(
                    f"The domain of {variable!r} cannot be empty."
                )
    
            try:
                unique_values = set(domain)
            except TypeError as error:
                raise TypeError(
                    f"Values in the domain of {variable!r} "
                    "must be hashable."
                ) from error
    
            if len(unique_values) != len(domain):
                raise ValueError(
                    f"The domain of {variable!r} must contain "
                    "unique values."
                )

    def add_edge(
        self,
        parent: str,
        child: str,
    ) -> None:
        """
        Add a directed dependency parent → child.
    
        The edge is rejected if it creates a cycle.
        """
    
        if parent not in self.graph:
            raise ValueError(
                f"Unknown parent variable: {parent!r}."
            )
    
        if child not in self.graph:
            raise ValueError(
                f"Unknown child variable: {child!r}."
            )
    
        if parent == child:
            raise ValueError(
                "A variable cannot be its own parent."
            )
    
        self.graph.add_edge(parent, child)
    
        if not nx.is_directed_acyclic_graph(self.graph):
            self.graph.remove_edge(parent, child)
    
            raise ValueError(
                f"Adding {parent!r} → {child!r} would "
                "create a directed cycle."
            )

    def parents(
        self,
        variable: str,
    ) -> tuple[str, ...]:
        """Return the parents of a variable."""
    
        if variable not in self.graph:
            raise ValueError(
                f"Unknown variable: {variable!r}."
            )
    
        return tuple(self.graph.predecessors(variable))
    
    
    def children(
        self,
        variable: str,
    ) -> tuple[str, ...]:
        """Return the children of a variable."""
    
        if variable not in self.graph:
            raise ValueError(
                f"Unknown variable: {variable!r}."
            )
    
        return tuple(self.graph.successors(variable))


    def topological_order(self) -> tuple[str, ...]:
        """Return a valid parent-before-child ordering."""
    
        return tuple(
            nx.topological_sort(self.graph)
        )

    def set_cpt(
        self,
        variable: str,
        factor: Factor,
    ) -> None:
        """
        Assign a conditional probability factor to a variable.
    
        The expected factor variable order is:
    
            [parent_1, parent_2, ..., variable]
    
        For every fixed parent assignment, the probabilities over the
        child variable must sum to one.
        """
    
        if variable not in self.graph:
            raise ValueError(
                f"Unknown variable: {variable!r}."
            )
    
        if not isinstance(factor, Factor):
            raise TypeError(
                "`factor` must be a Factor object."
            )
    
        self._validate_cpt(
            variable=variable,
            factor=factor,
        )
    
        # Store a copy so external modifications to the original factor
        # do not silently modify the Bayesian network.
        self.cpts[variable] = factor.copy()

    def _validate_cpt(
        self,
        variable: str,
        factor: Factor,
    ) -> None:
        """Validate one conditional probability factor."""
    
        expected_variables = [
            *self.parents(variable),
            variable,
        ]
    
        if factor.variables != expected_variables:
            raise ValueError(
                "The CPT variable order must be "
                f"{expected_variables}."
            )
    
        if np.any(factor.values < 0.0):
            raise ValueError(
                "CPT probabilities cannot be negative."
            )
    
        conditional_sums = factor.values.sum(axis=-1)
    
        if not np.allclose(
            conditional_sums,
            1.0,
        ):
            raise ValueError(
                f"The CPT for {variable!r} must sum to 1 "
                "over the child variable for every parent "
                "assignment."
            )
    def _validate_cpt(
        self,
        variable: str,
        factor: Factor,
    ) -> None:
        """
        Validate one conditional probability factor.
        """
    
        expected_variables = [
            *self.parents(variable),
            variable,
        ]
    
        if list(factor.variables) != expected_variables:
            raise ValueError(
                f"The CPT for {variable!r} must have variable order "
                f"{expected_variables}, but received "
                f"{list(factor.variables)}."
            )
    
        for factor_variable in expected_variables:
            expected_domain = self.domains[factor_variable]
            received_domain = factor.domains[factor_variable]
    
            if not np.array_equal(
                expected_domain,
                received_domain,
            ):
                raise ValueError(
                    f"The domain of {factor_variable!r} in the CPT "
                    f"does not match the Bayesian-network domain.\n"
                    f"Expected: {expected_domain.tolist()}\n"
                    f"Received: {received_domain.tolist()}"
                )
    
        if np.any(factor.values < 0.0):
            raise ValueError(
                "CPT probabilities cannot be negative."
            )
    
        if np.any(factor.values > 1.0):
            raise ValueError(
                "CPT probabilities cannot be greater than one."
            )
    
        conditional_sums = factor.values.sum(axis=-1)
    
        if not np.allclose(
            conditional_sums,
            1.0,
        ):
            raise ValueError(
                f"The CPT for {variable!r} must sum to 1 "
                "over the child variable for every parent "
                "assignment."
            )

    def validate(self) -> None:
        """Validate graph structure and all CPTs."""
    
        if not nx.is_directed_acyclic_graph(self.graph):
            raise ValueError(
                "The network graph must be acyclic."
            )
    
        missing_cpts = [
            variable
            for variable in self.graph.nodes
            if variable not in self.cpts
        ]
    
        if missing_cpts:
            raise ValueError(
                "CPTs are missing for: "
                f"{missing_cpts}."
            )
    
        for variable, factor in self.cpts.items():
            self._validate_cpt(
                variable,
                factor,
            )

    def joint_factor(self) -> Factor:
        """
        Multiply all CPTs to construct the full joint factor.
        """
    
        self.validate()
    
        ordered_variables = self.topological_order()
    
        result = self.cpts[
            ordered_variables[0]
        ].copy()
    
        for variable in ordered_variables[1:]:
            result = result * self.cpts[variable]
    
        result.name = "Joint Distribution"
    
        return result

    def probability(
        self,
        assignment: Mapping[str, Hashable],
    ) -> float:
        """
        Evaluate one complete joint assignment using the
        Bayesian-network factorization.
        """
    
        self.validate()
    
        missing_variables = [
            variable
            for variable in self.graph.nodes
            if variable not in assignment
        ]
    
        if missing_variables:
            raise ValueError(
                "The assignment is missing variables: "
                f"{missing_variables}."
            )
    
        unknown_variables = [
            variable
            for variable in assignment
            if variable not in self.graph
        ]
    
        if unknown_variables:
            raise ValueError(
                "The assignment contains unknown variables: "
                f"{unknown_variables}."
            )
    
        probability = 1.0
    
        for variable in self.topological_order():
            cpt = self.cpts[variable]
    
            local_assignment = {
                cpt_variable: assignment[cpt_variable]
                for cpt_variable in cpt.variables
            }
    
            probability *= cpt.get_value(
                local_assignment
            )
    
        return float(probability)

    def sample(
        self,
        size: int = 1,
        rng: np.random.Generator | None = None,
    ) -> list[dict[str, Hashable]]:
        """
        Generate samples using ancestral sampling.
        """
    
        self.validate()
    
        if not isinstance(size, (int, np.integer)):
            raise TypeError(
                "size must be an integer."
            )
    
        if size <= 0:
            raise ValueError(
                "size must be greater than zero."
            )
    
        if rng is None:
            rng = np.random.default_rng()
    
        samples = []
    
        for _ in range(size):
            assignment = {}
    
            for variable in self.topological_order():
                cpt = self.cpts[variable]
                parent_variables = self.parents(variable)
    
                if parent_variables:
                    reduced_cpt = cpt.reduce(
                        {
                            parent: assignment[parent]
                            for parent in parent_variables
                        }
                    )
                else:
                    reduced_cpt = cpt
    
                probabilities = np.asarray(
                    reduced_cpt.values,
                    dtype=float,
                )
    
                sampled_index = rng.choice(
                    len(self.domains[variable]),
                    p=probabilities,
                )
    
                assignment[variable] = (
                    self.domains[variable][sampled_index]
                )
    
            samples.append(assignment)
    
        return samples

    def plot(self) -> None:
        """Visualize the Bayesian-network graph."""
    
        plt.figure(figsize=(8, 5))
    
        positions = nx.spring_layout(
            self.graph,
            seed=42,
        )
    
        nx.draw_networkx(
            self.graph,
            pos=positions,
            arrows=True,
            node_size=3000,
            font_size=10,
        )
    
        plt.title(self.name)
        plt.axis("off")
        plt.tight_layout()
        plt.show()